# 머리 위에서 팔이 가려지는 두 장의 수행자 박스 다시 그리기

기본 검증의 **c04 78번과 c07 41번**을 다시 봅니다. 바꾸는 것은 수행자 박스의 기준 한 가지입니다. 먼저 두 원본을 판독하고 저장한 뒤, 지난 실행에서 저장한 원본 모델과 파인튜닝 모델의 예측에 대조합니다.

### 1. 판독 기준

- 선수의 **보이는 신체와 착용물(옷·신발)**을 모두 감싸도록 위·아래·좌·우 경계를 잡으세요. 머리 위로 손가락이나 손목 일부가 보이면 그 부분도 포함합니다.
- 손이 원판에 완전히 가려졌다면 **눈으로 확인되는 가장 높은 신체 부위**에서 윗변을 잡으세요. 바 높이로 손끝을 추정할 필요는 없습니다.
- 박스 안에 원판이 함께 들어가는 것은 자연스럽습니다. 상하좌우 끝을 정하는 대상은 수행자입니다. 두 장에 같은 기준을 쓰세요.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

from wlcv.annotation.performer_box import PerformerBox
from wlcv.annotation.pose_box_editor import visible_box_editor
from wlcv.config import ROOT, RUNS, VIDEOS
from wlcv.evaluation.performer_box_review import (
    FIRST_FOLDERS, REVIEW_FOLDER, compare_saved_boxes, draw_comparison,
)

scenes = list(FIRST_FOLDERS)
predictions = RUNS / "pose_first_finetune" / "manual_evaluation"
results = ROOT / "outputs" / "pose_visible_performer_review"

### 2. 두 탭에서 박스 그리기

각 탭의 원본 사진에서 한 모서리부터 대각선 반대편까지 **마우스로 드래그**하세요. 주황색 사각형을 확인하고 **박스 저장**을 누르면 초록색이 됩니다. 저장 전에는 다시 드래그해서 고칠 수 있습니다. 노트북을 다시 열면 이 판독에서 저장한 박스부터 이어서 편집할 수 있습니다.

In [ ]:
panels = []
for video, frame in scenes:
    second_box = PerformerBox(REVIEW_FOLDER, video, frame)
    panels.append(visible_box_editor(VIDEOS / f"{video}.mp4", frame, second_box))

tabs = widgets.Tab(children=panels)
for index, (video, frame) in enumerate(scenes):
    tabs.set_title(index, f"{video[:3]} {frame}번")
display(tabs)

### 3. 두 장 저장 후 비교

두 탭에서 저장한 다음 아래 셀을 실행하세요. 2차 박스는 `data/annotations/visible_performer_review/validation/person_boxes/`에 따로 저장됩니다.

표의 **윗변 변화(px)**가 양수이면 새 윗변이 원본 영상에서 더 아래에 있습니다. 모델별로 **동일한 예측 박스**를 고정하고 수동 박스만 바꿔 IoU(겹치는 면적의 비율)와 매칭 여부(IoU ≥ 0.50)를 비교합니다. 기존 후보를 그대로 쓰므로 IoU 변화는 **라벨 기준의 영향**을 보여줍니다. 무릎·발목 좌표는 이번 판독 대상에서 제외합니다.

In [ ]:
saved = sum(PerformerBox(REVIEW_FOLDER, video, frame).saved is not None for video, frame in scenes)
print(f"{saved}/{len(scenes)}장 저장")
if saved < len(scenes):
    print("두 탭에서 박스를 모두 저장한 뒤 이 셀을 다시 실행하세요.")
else:
    report = compare_saved_boxes(REVIEW_FOLDER, predictions)
    results.mkdir(parents=True, exist_ok=True)
    report.to_csv(results / "box_comparison.csv", index=False)
    display(report.drop(columns=["1차 박스", "2차 박스", "예측 박스"]).round(3))
    figure = draw_comparison(report)
    figure.savefig(results / "box_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"표와 그림 저장 위치: {results}")